# 02 — Feature Engineering
### Credit Card Fraud Detection

This notebook turns the decisions from `01_eda.ipynb` into the actual data
preparation recipe used from here on:

1. Apply the EDA-informed duplicate decision and finalize the train/test split
2. Run the `src/features.py` preprocessing pipeline end-to-end and verify it
   behaves correctly (fit on train only, no leakage into test)
3. Quantify the two imbalance-handling strategies (class weights vs. SMOTE)
   and build visual intuition for what SMOTE actually does, before Phase 7
   wires either strategy into real cross-validated training

**Important scope note:** this notebook does *not* persist a fitted
preprocessor or a SMOTE-resampled dataset to disk. Fitting must happen
inside the training pipeline (ideally per-CV-fold) to avoid leakage — that
happens in `03_modeling.ipynb`. This notebook validates the *recipe*, not
the final artifact.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from src.config import TARGET_COL, RANDOM_STATE, FIGURES_DIR
from src.data import load_raw_data, train_test_split_stratified, save_processed_data
from src.features import (
    add_engineered_features,
    build_preprocessor,
    get_feature_names_out,
    get_class_weights,
    get_scale_pos_weight,
    apply_smote,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", 50)

%matplotlib inline


## 1. Recap EDA Decision: Duplicate Rows

From `01_eda.ipynb` section 2: this dataset contains fully duplicated rows
(identical across all 31 columns, including `Class`). These are treated as
genuine repeated data-entry artifacts, not signal — a model that memorizes
a duplicated fraud row and sees it again in the test set would get an
inflated, misleading evaluation. **We drop full duplicates before splitting**,
keeping the first occurrence of each.

In [ ]:
df = load_raw_data()
print(f"Before dedup: {df.shape}")

n_dupes = df.duplicated().sum()
print(f"Dropping {n_dupes} fully duplicated rows "
      f"({n_dupes / len(df) * 100:.3f}% of dataset)")

df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"After dedup:  {df.shape}")

fraud_pct_before_after = df[TARGET_COL].mean() * 100
print(f"Fraud rate after dedup: {fraud_pct_before_after:.4f}%")


## 2. Finalize Train/Test Split

Re-running the stratified split (same logic as Phase 4's `make prepare-data`,
now on the deduplicated data) and **overwriting** `data/processed/train.csv`
and `data/processed/test.csv`. From this point forward, every notebook and
the training script load this version — it's the canonical split for the
project.

> If you ever want the non-deduplicated baseline back for comparison, it's
> reproducible via `make prepare-data` on the raw file directly.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split_stratified(df)

print(f"Train: {X_train.shape}, fraud rate: {y_train.mean() * 100:.4f}%  ({y_train.sum()} positives)")
print(f"Test:  {X_test.shape}, fraud rate: {y_test.mean() * 100:.4f}%  ({y_test.sum()} positives)")

save_processed_data(X_train, X_test, y_train, y_test)


## 3. Apply Feature Engineering

`add_engineered_features()` is stateless (no fitting, just deterministic
transforms), so it's safe to apply independently to train and test without
any leakage risk — unlike the scaler in the next section, which must be fit
on train only.

In [ ]:
X_train_fe = add_engineered_features(X_train)
X_test_fe = add_engineered_features(X_test)

X_train_fe[["Time", "Amount", "Amount_log", "Hour"]].head()


## 4. Preprocessing: Fit on Train Only

`build_preprocessor()` returns a `ColumnTransformer` that:
- Applies `RobustScaler` to `Time`, `Amount`, `Amount_log`
- Passes `V1`-`V28` through untouched (already ~standardized by the
  original PCA upstream)
- Passes `Hour` through untouched

We **fit on `X_train_fe` only**, then use that same fitted transformer to
transform `X_test_fe`. Fitting on the full dataset (train+test combined)
before splitting is a classic leakage mistake — the scaler would "see" the
test set's distribution during fitting, giving an overly optimistic
evaluation later. We deliberately avoid that here.

In [ ]:
preprocessor = build_preprocessor()

X_train_processed = preprocessor.fit_transform(X_train_fe)
X_test_processed = preprocessor.transform(X_test_fe)   # transform only, never fit

feature_names = get_feature_names_out(preprocessor)

print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")
print(f"Feature order: {feature_names}")


### Verify V1-V28 passed through unchanged (sanity check on the pipeline itself)

In [ ]:
v1_idx = feature_names.index("V1")
v1_before = X_train_fe["V1"].values
v1_after = X_train_processed[:, v1_idx]

unchanged = np.allclose(v1_before, v1_after)
print(f"V1 unchanged by preprocessing: {unchanged}")
assert unchanged, "V1 should pass through the ColumnTransformer untouched"


### Visualize the scaling effect on Time / Amount / Amount_log

In [ ]:
scaled_cols = ["Time", "Amount", "Amount_log"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, col in enumerate(scaled_cols):
    axes[0, i].hist(X_train_fe[col], bins=50, color="steelblue")
    axes[0, i].set_title(f"{col} (before scaling)")

    col_idx = feature_names.index(col)
    axes[1, i].hist(X_train_processed[:, col_idx], bins=50, color="darkorange")
    axes[1, i].set_title(f"{col} (after RobustScaler)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_scaling_before_after.png", bbox_inches="tight")
plt.show()


**Why `RobustScaler` and not `StandardScaler`:** `RobustScaler` centers
using the median and scales using the interquartile range, rather than mean
and standard deviation. Given the extreme outliers in `Amount` (visible in
notebook 01's boxplots), a mean/std-based scaler would be dominated by those
few large-value transactions — the median/IQR approach is far less sensitive
to them.

## 5. Class Imbalance: Quantifying Both Strategies

`src/models.py` (Phase 4) supports two ways of handling the 0.17% positive
rate, and Phase 7 will compare both empirically. Here we just compute what
each one actually looks like on this training set.

In [ ]:
weights = get_class_weights(y_train)
spw = get_scale_pos_weight(y_train)

print("Strategy A — class_weight dict (sklearn: LogisticRegression, RandomForest):")
print(f"  {weights}")
print()
print("Strategy B — scale_pos_weight scalar (XGBoost / LightGBM / CatBoost convention):")
print(f"  {spw:.2f}  (i.e. treat each positive example as ~{spw:.0f}x a negative one)")


## 6. SMOTE: Building Intuition

The third strategy — oversampling the minority class — works differently
from the two above: instead of *reweighting* how much each example counts
during training, SMOTE generates synthetic fraud examples by interpolating
between real fraud examples and their nearest fraud neighbors in feature
space.

> ⚠️ **Leakage warning, stated explicitly:** the SMOTE call below is applied
> to the *entire* processed training set, once, purely to visualize what it
> does. **This is not how it will be used for actual model training.** In
> Phase 7, SMOTE will be applied *inside* each cross-validation fold (via an
> `imblearn.pipeline.Pipeline`), so that synthetic points are only ever
> generated from that fold's training portion — never from data that will
> be used to evaluate that fold. Applying SMOTE once before CV would leak
> information about validation-fold neighbors into the synthetic training
> points, inflating CV scores in a way that won't hold up on the real test
> set.

In [ ]:
X_train_smote, y_train_smote = apply_smote(
    pd.DataFrame(X_train_processed, columns=feature_names),
    y_train.reset_index(drop=True),
    sampling_strategy=0.1,
)

print("Before SMOTE:")
print(y_train.value_counts().rename({0: "Legitimate", 1: "Fraud"}))
print("\nAfter SMOTE (sampling_strategy=0.1 -> fraud oversampled to 10% of legitimate count):")
print(y_train_smote.value_counts().rename({0: "Legitimate", 1: "Fraud"}))


### Visualize real vs. synthetic fraud points in a 2D projection

For visualization only — a 2D PCA projection of the (already 30-dimensional) processed features, so we can see roughly where SMOTE's synthetic points land relative to the real distribution.

In [ ]:
viz_pca = PCA(n_components=2, random_state=RANDOM_STATE)
viz_pca.fit(X_train_processed)

# Sample legitimate points for a readable scatter plot (plotting 227k points is unreadable and slow)
rng = np.random.RandomState(RANDOM_STATE)
legit_sample_idx = rng.choice(np.where(y_train.values == 0)[0], size=3000, replace=False)

before_2d_legit = viz_pca.transform(X_train_processed[legit_sample_idx])
before_2d_fraud = viz_pca.transform(X_train_processed[y_train.values == 1])

# In the SMOTE-resampled set, the original fraud rows come first (imblearn preserves original
# row order, appending synthetic rows after), so we can split real vs. synthetic this way.
n_original_fraud = int(y_train.sum())
fraud_rows_smote = X_train_smote[y_train_smote == 1].values
after_2d_fraud_real = viz_pca.transform(fraud_rows_smote[:n_original_fraud])
after_2d_fraud_synthetic = viz_pca.transform(fraud_rows_smote[n_original_fraud:])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(before_2d_legit[:, 0], before_2d_legit[:, 1], s=8, alpha=0.3, label="Legitimate (sample)")
axes[0].scatter(before_2d_fraud[:, 0], before_2d_fraud[:, 1], s=12, alpha=0.8, color="crimson", label="Fraud (real)")
axes[0].set_title("Before SMOTE")
axes[0].legend()

axes[1].scatter(before_2d_legit[:, 0], before_2d_legit[:, 1], s=8, alpha=0.3, label="Legitimate (sample)")
axes[1].scatter(after_2d_fraud_real[:, 0], after_2d_fraud_real[:, 1], s=12, alpha=0.8, color="crimson", label="Fraud (real)")
axes[1].scatter(after_2d_fraud_synthetic[:, 0], after_2d_fraud_synthetic[:, 1], s=10, alpha=0.5, color="orange", label="Fraud (synthetic)")
axes[1].set_title("After SMOTE (sampling_strategy=0.1)")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "08_smote_visualization.png", bbox_inches="tight")
plt.show()


**What to look for:** synthetic points (orange) should sit *between* real
fraud points (crimson) — that's SMOTE's interpolation mechanism working as
intended. If synthetic points instead land in the middle of the legitimate
cloud, it usually means the fraud class isn't well clustered in this
feature space, and SMOTE's nearest-neighbor interpolation is bridging
across meaningfully different fraud "types" — worth flagging as a modeling
risk to watch for in Phase 8's SHAP analysis.

## 7. Key Decisions Feeding Into Phase 7



1. **Duplicates:** dropped `N` rows pre-split (see section 1 output) — new
   canonical split saved to `data/processed/train.csv` / `test.csv`.
2. **Scaling:** `RobustScaler` on `Time`/`Amount`/`Amount_log` confirmed
   working correctly; `V1`-`V28` confirmed passed through unchanged.
3. **Class weight strategy:** `class_weight` dict and `scale_pos_weight`
   scalar both computed above — Phase 7 will compare models using each
   against SMOTE-augmented training.
4. **SMOTE:** confirmed working as expected on this feature space (or:
   _note here if the 2D visualization looked concerning_). Reminder for
   Phase 7: **SMOTE must be applied inside each CV fold**, never once on
   the full training set before cross-validation.
5. **Open question to resolve empirically in Phase 7:** does SMOTE
   meaningfully outperform class-weighting on PR-AUC for this dataset, or
   does it just add training time for a similar (or worse) result? Don't
   assume the fancier technique wins — compare both.
